# Mobile apps reviews classification with spaCy and sklearn (Bag of Words and Decision Trees)

This notebook illustrates text classification using **Bag od Words (BoW)** for text representation and **Decision Trees (DT)** algorithm for classification. <br>

The classification task is classification of mobile app reviews into:
* functional requirements,
* non-functional requirements,
* other (reviews irrelevant from Requirements Engineering perspective)

The dataset used in the notebook originates from the paper [Listening to the Crowd for the Release Planning of Mobile Apps](https://dl.acm.org/doi/abs/10.1109/TSE.2017.2759112). It consists of manually classified 3,000 user reviews of different Android apps.<br>

### Load the required libraries

In [ ]:
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import metrics

from sklearn.tree import DecisionTreeClassifier

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import matplotlib.pyplot as plt
import seaborn as sb

import spacy

from IPython.display import display

from pathlib import Path
import pickle
import json

In [ ]:
RAND_STATE = 1

### Load and explore the data

In [ ]:
data = pd.read_csv(Path.cwd() / 'data' / 'mobile_apps_reviews.csv')

Loading the data in case of running this code from Google Colab

In [ ]:
# from google.colab import files
#
# data_file = files.upload()

# The uploaded object is a dictionary with the filename and the file content as key-value pairs - in this case just one such pair.
# We extract the file name and read in the file content using pandas (to have it in the DataFrame format)
# file_name = list(data_file.keys())[0]
# data = pd.read_csv(file_name)

In [ ]:
data.head(10)

In [ ]:
data.info()

Examine the values of the outcome variable

In [ ]:
data.category.value_counts()

Group these values (as suggested in the paper the data originates from) so that we have 3 possible outcome values:
* functional: BUG, FEATURE
* non-functional: PERFORMANCE, USABILITY, ENERGY, SECURITY
* other: OTHER

In [ ]:
def category_to_cls(cat):
  if cat == "OTHER": return cat
  return "FUNCTIONAL" if cat in ['BUG', 'FEATURE'] else "NON-FUNCTIONAL"

data['cls'] = data.category.apply(category_to_cls)

Examine how balanced the dataset is

In [ ]:
data.cls.value_counts(normalize=True)

Considering the lack of balance in the outcome variable, we might need to apply a balancing method when building a classification model

Examine a few reviews

In [ ]:
for i, review in enumerate(data.sample(n=10, random_state=RAND_STATE).review):
    print(f"{i+1}) {review}\n\n")

Examine the length of the reviews.

In [ ]:
data['review_len'] = data.review.apply(len)
data.review_len.describe()

In [ ]:
plt.figure(figsize=(10, 6))

# sb.boxplot(data=data, x='cls', y='review_len', hue='cls', fill=True, saturation=0.75, width=0.55, gap=0.25, linewidth=2, legend=False)
# too many outliers, difficult to examine

sb.kdeplot(data=data,
           x='review_len',
           hue='cls',
           fill=True,
           # ensures that each kde curve is normalized independently so that the area under each individual group's curve sums to 1;
           # suitable for comparing distributions across the groups
           common_norm=False,
           alpha=0.5,
           linewidth=2)

plt.title('Distribution of review length across review categories')
plt.xlabel('Review length')
plt.ylabel('Density')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


Examine also review ratings

In [ ]:
plt.figure(figsize=(8, 6))

sb.boxplot(data=data, x='cls', y='rating', hue='cls', fill=True, saturation=0.75, width=0.55, gap=0.25, linewidth=2, legend=False)
# sb.kdeplot(data=data, x='rating', hue='cls', fill=True, common_norm=False, alpha=0.5, linewidth=2)

plt.title('Distribution of review ratings across review categories')
plt.ylabel('Review rating')
plt.xlabel('')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

Both review length and rating seem to be relevant predictors of the outcome variable. So, we will keep these two variables in adddition to the review text and the outcome variables. The rest can be dropped

In [ ]:
data_sub = data[['rating', 'review_len', 'review', 'cls']]

In [ ]:
data_sub.head(10)

### Text preprocesing (using Spacy)

[`spaCy`](https://spacy.io/) is an open-source natural language processing library for Python. It is designed particularly for production use, and thus can be used to build applications that process massive volumes of text efficiently.

First, load spacy's basic English language model (we use here the small one: `en_core_web_sm`). This is, in fact, a pipeline of software componets for processing English language texts. A brief overview is given [here](https://spacy.io/usage/processing-pipelines) <br>

Note: In addition to English, spaCy supports several other languages. See [this page](https://spacy.io/usage/models) of the spacy's documentation to epxlore the available language models.

In [ ]:
nlp_pipe = spacy.load('en_core_web_sm')

Note: if the previous line ends in an error, it is because spacy's `en_core_web_sm` language model is not available. <br>To download it, execute the following line in the command prompt or terminal: <br>
`python -m spacy download en_core_web_sm` <br>

The Spacy's English pipeline that we have loaded (`en_core_web_sm`) consists of several text processing components including: *tok2vec*, *tagger*, *parser*, *lemmatizer*, and *ner*. For more information about this particular pipeline and the other available ones, see the [documentation](https://spacy.io/models/en).


#### Explore Spacy a bit

Let's explore what spacy provides us with using one randomly selected review

In [ ]:
sample_review = data_sub.sample(random_state=RAND_STATE)['review']
sample_review_cls = data_sub.sample(random_state=RAND_STATE)['cls']

In [ ]:
print(f"Review text:\n{sample_review.iloc[0]}")
print(f"\nReview class:\n{sample_review_cls.iloc[0]}")

Take the text of the sample review and pass it to spacy for processing

In [ ]:
sample_review = sample_review.iloc[0]

parsed_review = nlp_pipe(sample_review)

Examine individual tokens and their linguistic features (see the explanations and examples in the [spacy's documentation](https://spacy.io/usage/linguistic-features)).  

Let's explore a subset of the most frequently used these features:
* each POS (Part of Speech), representing the token's role in the sentence; a widely used POS tags can be seen [here](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html)
* lemma: the main (dictionary) word form

In [ ]:
token_text = [token.text for token in parsed_review]
token_pos = [token.pos_ for token in parsed_review]
token_lemma = [token.lemma_ for token in parsed_review]

pd.DataFrame(zip(token_text, token_pos, token_lemma), columns=['text', 'POS', 'lemma']).head(15)

We can also visualise the results of the sentence parsing

In [ ]:
from spacy import displacy

displacy.render(parsed_review, style="dep", jupyter=True)

# an alternative, if more compact display is preferred
# displacy.render(parsed_review, style="dep", jupyter=True, options={'compact':True})


Examine stop words

In [ ]:
spacy_stopwords = spacy.lang.en.stop_words.STOP_WORDS

# the total number of stop words:
print(f'Number of stop words: {len(spacy_stopwords)}\n')

# print first ten stop words:
print(f'First 20 stop words:\n{", ".join(sorted(list(spacy_stopwords))[:20])}')

In [ ]:
print(f"Is 'not' in the spacy's stop words list: {'Yes' if 'not' in spacy_stopwords else 'No'}")

Do the "cleaning" of the sample review by removing punctuation marks, spaces, and numbers; remove also stop words, except for the negation ('not')

In [ ]:
spacy_stopwords.discard('not')
filtered_tokens = [token.lemma_ for token in parsed_review
 if (not token.is_punct) and (not token.is_space) and (not token.like_num) and (token.lemma_.lower() not in spacy_stopwords)]

print(f"Number of tokens before preprocessing: {len(parsed_review)}")
print(f"Number of tokens after preprocessing: {len(filtered_tokens)}")

In [ ]:
print(f"Original sample review:\n{sample_review}")
print(f"Sample review after pre-processing:\n{' '.join(filtered_tokens)}")

#### Use spacy to preprocess all reviews

Now that we have seen what spacy can do for text preprocessing, let's now use its functionalities to create a function for custom preprocessing of app reviews:

In [ ]:
def spacy_preprocessing(doc):
    # Create tokens object with linguistic annotations
    tokens = nlp_pipe(doc)

    # remove punctuation, numbers, and URLs
    tokens = [token for token in tokens if ((not token.is_punct) and (not token.is_space) and (not token.like_num) and (not token.like_url))]

    # remove stopwords
    tokens = [token for token in tokens if (not token.is_stop) or (token.lemma_.lower() == 'not')]

    # Lemmatize each token and convert each token into lowercase
    words = [token.lemma_.lower() for token in tokens]

    # return preprocessed list of words
    return words

In [ ]:
sample_tokens = spacy_preprocessing(sample_review)
print(f"Sample review after preprocessing:\n{' '.join(sample_tokens)}\n")
print(f"Total number of words after preprocessing: {len(sample_tokens)}" )

### Text transformation (using BoW)

Create counts-based vectoriser that will, for the given set of documents (reviews), build a Bag of Words (BoW) model.

It can be easily created as an instance of the scikit-learn's [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) class

In [ ]:
def create_count_vectoriser(ngrams=(1,1), min_doc_freq=0.01, f_max=-1):
  vectoriser = CountVectorizer(
      tokenizer = spacy_preprocessing, # note the use of the above defined spacy_preprocessing f.
      token_pattern=None, # we use our own way of splitting text into tokens and letting the vectoriser to know it
      ngram_range=ngrams,
      max_features=f_max if f_max > 0 else None,
      min_df=min_doc_freq
  )
  return vectoriser

Let's see how it works before we apply it in the model building pipeline:

In [ ]:
vectorizer_example = create_count_vectoriser()
vectorised_reviews = vectorizer_example.fit_transform(data_sub.review)
pd.DataFrame(data=vectorised_reviews.toarray(), columns=vectorizer_example.get_feature_names_out())

### Create, fit, and test text classification pipeline

We will create a review classifier as a sequence ('pipeline') of several components, using the [sklearn's Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

**Pipeline** is an auxiliary sklearn's class that chains together multiple processing steps (called estimators or transformers) into a single, unified computational object. It can be considered as an assembly line for data:
* Raw data serves as the input
* It passes sequentially through a series of processing steps (e.g., cleaning, scaling, fine tuning).
* A trained classification / regression model comes out at the very end.

Why do we need / use it?
* Cleaner code, easier to read and maintain
* Prevention of data leakage, as the Pipeline ensures that the fit() and transoform() methods for every step in the processing sequence (e.g., a scaler, a classifier) are only called on the training data
* Seamless fine-tuning of hyper-parameters of the entire processing sequence (preprocessing + model).


To build a Pipeline, first, we split the data into train/test sets:

In [ ]:
X = data_sub[['review','review_len','rating']]
y = data_sub.cls.values

# do label encoding
y_encodings, y_levels = pd.factorize(y)

In [ ]:
display(y_levels)
pd.Series(y_encodings).value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y_encodings,
                                                    test_size = 0.2,
                                                    random_state = RAND_STATE,
                                                    stratify=y_encodings)

We will use the [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) class to apply different data transformations to different columns in the dataset.

This class is particularly useful when dealing with heterogeneous data, where some columns might be numerical and require scaling, while others are categorical and need encoding or textual and require special processing (as here).

In [ ]:
# Instantiate the vectoriser
count_vectorizer = create_count_vectoriser()

# Apply the vectoriser to the 'review' column and add the other two columns (rating and review length) as is
column_transformer = ColumnTransformer(
    [('word_counts', count_vectorizer, 'review')],
    remainder='passthrough'
    )

As the classifier, we will use **Decision Tree (DT)** classifier (implemented in sklearn's [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) class).

In [ ]:
classifier = DecisionTreeClassifier(random_state=RAND_STATE, min_impurity_decrease=0.001)

We can now define the classification pipeline:

In [ ]:
pipe = Pipeline([('vectorizer', column_transformer),
                 ('classifier', classifier)])

# Fit the pipepline to the data from the training set
pipe.fit(X_train, y_train)

A few notes about the accessing elements of a Pipeline:

`pipe.named_steps` provides a dictionary of the Pipeline's structure

In [ ]:
pipe.named_steps

To access any individual component of a Pipeline, use `named_steps` followed by the name of the component. For example, to see the fitted parameters of the decision tree classifier, we can do the following:

In [ ]:
pipe.named_steps['classifier'].get_params()

### Evaluate the model on the test data

First, examine the confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay.from_estimator(
        pipe,
        X_test,
        y_test,
        display_labels=y_levels,
        cmap='Blues',
    )

Now, we can compute evaluation measures:

Precision = TP/(TP + FP) <br>
Recall = TP/(TP + FN) <br>
Accuracy = (TP + TN) / m <br>
F1 = 2 x precision x recall/(precision + recall)

In [ ]:
predictions_1 = pipe.predict(X_test)

report_1 = metrics.classification_report(y_true = y_test, y_pred = predictions_1)
print(report_1)

In [ ]:
list(y_levels)

Note:

*   `support` is the number of occurrences of each class in y_true
*   `macro avg` is the average of unweighted score per label (i.e. class)
*   `weighted avg` is the average of support-weighted score per label (i.e. class)

Macro vs weighted average:

* **Macro average** treats all classes equally, no matter their size; therefore, poor performance on minority or difficult classes significantly affects its value
* **Weighted average** accounts for class imbalance (more frequent classes contribute more to the final score); it is used when performance across dominant classes is prioritized over performances on rare classes

Since the dataset is imbalanced, we also compute [balanced accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html). It is  calculated as the sum of accuracies per class (= recall) divided by the total number of test samples.

It is used when we care equally about every class and want to ensure the model doesn't achieve a high overall score while performing terribly on the minority class(es).

In [ ]:
display(f"Accuracy: {metrics.accuracy_score(y_test, predictions_1):0.4f}" )
display(f"Balanced accuracy: {metrics.balanced_accuracy_score(y_test, predictions_1, adjusted=True):0.4f}" )

### Try to improve the classifer through hyperparameters tuning

We will now try to improve the performance of the classifier by changing the way text transformation is done and the complexity of the decision tree.

Often used text transformation hyperparameters include:
* min_df - minimum document frequency
* ngram_range - ngrams that are used as features

We can try both, but that will take a long time to execute, so we'll focus on one and determine optimal value through *Cross-Validation*, using the [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) class.

Note: hyperparamter names have to be preceded by the name of the corresponding component (transformer or estimator) in the pipeline; in this case, names of the relevant pipeline components are `vectorizer` and `classifier`. However, as the 'vectorizer' is not a single transformer but may contain multiple transformers (one per column), we need to specify the name of the particular transformer we want to fine-tune - in this case, it is 'word_counts'. So, we have to precede the name of the text transformation hyperparameter with `vectorizer__word_counts__`.

In [ ]:
%%time

param_grid = {
    'vectorizer__word_counts__min_df':[0.005, 0.01, 0.025],
    'classifier__min_impurity_decrease': [0.001, 0.005, 0.01]
}
grid_search = GridSearchCV(pipe,
                           param_grid=param_grid,
                           scoring='f1_macro',
                           verbose=2)
grid_search.fit(X_train, y_train)

In [ ]:
display('Best hyperparameters: ', grid_search.best_params_)
display('Best scores: ', grid_search.best_score_)

Re-run the pipeline with the optimal hyperparamter values:

In [ ]:
best_min_df = grid_search.best_params_['vectorizer__word_counts__min_df']
best_min_impurity_dec = grid_search.best_params_['classifier__min_impurity_decrease']

# note how we configure the pipeline with a specific value for a hyperparameter
pipe.set_params(vectorizer__word_counts__min_df=best_min_df,
                  classifier__min_impurity_decrease=best_min_impurity_dec)
pipe.fit(X_train,y_train)

Make predictions with the fine-tuned pipeline and evaluate the performance

In [ ]:
predictions_2 = pipe.predict(X_test)

report_2 = metrics.classification_report(y_true=y_test, y_pred=predictions_2)
print(report_2)

In [ ]:
print(f"Balanced accuracy: {metrics.balanced_accuracy_score(y_test, predictions_2, adjusted=True):0.4f}" )

Save the best model as well as its results

In [ ]:
results_dir = Path.cwd() / 'results'
if not results_dir.exists():
    results_dir.mkdir()

with open(results_dir / 'BoW_best_model.pkl', 'wb') as fobj:
    pickle.dump(pipe, fobj)

with open(results_dir / 'BoW_best_model_scores.json', 'w') as fobj:
    model_scores = metrics.classification_report(y_true=y_test, y_pred=predictions_2, output_dict=True)
    json.dump(model_scores, fobj, indent=4)